In [ ]:
import torch
from torch.nn import Linear
from torch_geometric.nn import TransformerConv
import coremltools

scikit-learn version 1.8.0 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.11.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/coremltools/optimize/torch/palettization/fake_palettize.py:82: SyntaxWarning: invalid escape sequence '\_'
  n_bits (:obj:`int`): Number of palettization bits. There would be :math:`2^{n\_bits}` unique weights in the ``LUT``.


In [2]:
class EdgeClassifier(torch.nn.Module):
    def __init__(self, in_channels, edge_dim):
        super().__init__()
        # Concatenate src_node, dst_node, and raw edge_attr
        self.lin = Linear(in_channels * 2 + edge_dim, in_channels)
        self.lin_final = Linear(in_channels, 1)

    def forward(self, z_src, z_dst, edge_attr):
        h = torch.cat([z_src, z_dst, edge_attr], dim=-1)
        h = self.lin(h).relu()
        return self.lin_final(h)

In [ ]:
model = EdgeClassifier(in_channels=16, edge_dim=8)
example_input = (torch.randn(1, 16), torch.randn(1, 16), torch.randn(1, 8))
traced_model = torch.jit.trace(model, example_input)
coreml_model = coremltools.convert(traced_model, inputs=[coremltools.TensorType(shape=example_input[0].shape), coremltools.TensorType(shape=example_input[1].shape), coremltools.TensorType(shape=example_input[2].shape)])
coreml_model.save("mlpackages/EdgeClassifier.mlpackage")

When both 'convert_to' and 'minimum_deployment_target' not specified, 'convert_to' is set to "mlprogram" and 'minimum_deployment_target' is set to ct.target.iOS15 (which is same as ct.target.macOS12). Note: the model will not run on systems older than iOS15/macOS12/watchOS8/tvOS15. In order to make your model run on older system, please set the 'minimum_deployment_target' to iOS14/iOS13. Details please see the link: https://apple.github.io/coremltools/docs-guides/source/target-conversion-formats.html
Model is not in eval mode. Consider calling '.eval()' on your model prior to conversion
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 10507.65 passes/s]


In [ ]:
class GraphAttentionEmbedding(torch.nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        edge_dim = msg_dim + time_enc.out_channels
        self.conv = TransformerConv(in_channels, out_channels // 2, heads=2,
                                    dropout=0.1, edge_dim=edge_dim)

    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = last_update[edge_index[0]] - t
        rel_t_enc = self.time_enc(rel_t.to(x.dtype))
        edge_attr = torch.cat([rel_t_enc, msg], dim=-1)
        return self.conv(x, edge_index, edge_attr)

Help on module coremltools:

NAME
    coremltools

FILE
    /Users/darrellcr/Devs/apple_aiml/c1/tabular/python/coremltools.py




In [ ]:
gnn = GraphAttentionEmbedding(
    in_channels=memory_dim,
    out_channels=embedding_dim,
    msg_dim=data.msg.size(-1),
    time_enc=memory.time_enc,
)